In [92]:
from itertools import product
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

In [93]:
def interpolate_small_gaps_series(s, freq, threshold, method='time'):
    """
    s: pandas Series with a DatetimeIndex (may be irregular)
    Only interpolates gaps whose total span <= threshold.
    """
    td = pd.Timedelta(threshold)
    s = s.copy()

    # ensure datetime index and handle duplicate timestamps
    if not isinstance(s.index, pd.DatetimeIndex):
        s.index = pd.to_datetime(s.index, format='ISO8601')
    s = s.sort_index()
    if not s.index.is_unique:
        s = s.groupby(level=0).mean()   # or .last(), .sum(), as appropriate

    # put on regular grid
    s = s.asfreq(freq)

    print(s)

    # gap span across each NaN run
    prev_valid_time = s.index.to_series().where(s.notna()).ffill()
    next_valid_time = s.index.to_series().where(s.notna()).bfill()
    span = next_valid_time - prev_valid_time

    # interpolate only small gaps
    filled = s.interpolate(method=method, limit_area='inside')
    return filled.where((span <= td) | s.notna())

In [126]:
def process(df, df_deltas, ts_col="_time", v_col="power"):
   gaps = df_deltas.copy()
   gaps[ts_col] = pd.to_datetime(gaps[ts_col], utc=True, format='ISO8601')
   gaps = gaps[[ts_col, "delta_gt_mean"]].sort_values(ts_col)

   # previous interval boundary (left-open)
   gaps["prev_time"] = gaps[ts_col].shift(1)

   # --- 1) Ensure both sides use UTC-aware datetimes for the merge keys
   df[ts_col] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")

   # Distrubute values on resampled grid
   cum = df[v_col].fillna(0).cumsum().where(df[v_col].notna())
   df2 = df[[ts_col]].copy()
   df2["cum"] = cum

   gaps[ts_col] = pd.to_datetime(gaps[ts_col], utc=True, errors="coerce")

   merged = pd.merge_asof(
      df2.sort_values(ts_col),
      gaps.sort_values(ts_col),
      left_on=ts_col,
      right_on=ts_col,
      direction="forward",  # pick the interval that ends at the next _time >= ts
      allow_exact_matches=True,
   )

   merged.to_csv("Temp.csv")

   # Inside the interval iff prev_time < ts <= _time
   left_boundary = merged["prev_time"].fillna(pd.Timestamp.min.tz_localize("UTC"))
   in_interval = merged[ts_col].gt(left_boundary) & merged[ts_col].le(merged[ts_col])

   # Flag rows by the interval’s delta_gt_mean; rows not in any interval -> NaN/False
   merged["in_big_gap"] = np.where(in_interval, merged["delta_gt_mean"], np.nan)

   # --- 4) Apply your conditional transform on `other` ---
   other_out = df2.copy()
   # Example: tag rows and (optionally) modify values when inside a "big gap"
   other_out["delta_gt_mean"] = merged["in_big_gap"].fillna(False)
   
   other_out.to_csv("outt.csv")

   # Example transform: set value to NaN during big gaps
   # other_out.loc[other_out["delta_gt_mean"], "value"] = np.nan

   mask = other_out["delta_gt_mean"]
   if mask.dtype not in ("bool", "boolean"):
      mask = (pd.Series(mask)
               .astype(str).str.strip().str.lower()
               .map({"true": True, "false": False, "1": True, "0": False}))
   mask = mask.fillna(False).astype("boolean")

   s_masked = other_out.where(~mask)    # keep values; NaN where big gaps
   other_out["lin_interpolate"] = s_masked["cum"].interpolate(limit_area="inside").ffill()  

   other_out.to_csv("out.csv")

   return other_out

In [128]:
def cumulative_linear_fill(freq: str, device: str):
    
    infile = Path(f"../data/things/{device}.csv")
    infile_deltas = Path(f"../data/resampled-{freq}/{device}_time_deltas.csv")
    outfile = Path(f"../data/resampled-{freq}/cumulinfilled/{device}.csv")

    outfile.parent.mkdir(parents=True, exist_ok=True)  # create folders if missing

    if infile.exists() and infile.is_file():
        if outfile.exists() and outfile.is_file():
            print(f"Infill for {device} at freq {freq} performed earlier. Skipping.")
        else:
            df = pd.read_csv(
                infile,
                # index_col="_time",
                parse_dates=True,
                date_format="ISO8601",
            )
            df_deltas = pd.read_csv(
                infile_deltas,
                # index_col="_time",
                parse_dates=True,
                date_format="ISO8601",
            )
            # mean_delta = pd.to_timedelta(time_stats.iloc[[3]]['_time']) # larger deltas are not filled
            # df["power"].plot()
            df = df[["_time", "power"]]
            df["power"] = process(df, df_deltas)["lin_interpolate"]

            outfile.parent.mkdir(exist_ok=True, parents=True)
            df.to_csv(outfile)
    else:
        print(f"No resampled data for {device} at freq {freq}. Skipping.")

In [ ]:
def reverse_cumulative_sum(freq: str, device: str):
    infile = Path(f"../data/resampled-{freq}/cumulinfilled/{device}.csv")
    outfile = Path(f"../data/resampled-{freq}/final/{device}.csv")

    outfile.parent.mkdir(parents=True, exist_ok=True)  # create folders if missing

    if infile.exists() and infile.is_file():
        if outfile.exists() and outfile.is_file():
            print(f"Infill for {device} at freq {freq} performed earlier. Skipping.")
        else:
            df = pd.read_csv(
                infile,
                # index_col="_time",
                parse_dates=True,
                date_format="ISO8601",
            )

            df["power"] = df["power"].diff().fillna(df["power"].iloc[0]) 

            df["power"].plot()

            outfile.parent.mkdir(exist_ok=True, parents=True)
            df.to_csv(outfile)
    else:
        print(f"No resampled data for {device} at freq {freq}. Skipping.")

In [130]:
for freq, device in tqdm(list(product(
    [
        # "15min",
        # "1min",
        "10s",
    ],
    [
        "DECT210 Waschmaschine",
        "DECT200 Waschmaschine",
        "Smart Switch 6 Spülmaschine",
        "DECT200 Spülmaschine",
    ]
))):
    # cumulative_linear_fill(freq, device)
    reverse_cumulative_sum(freq, device)

  0%|          | 0/4 [00:00<?, ?it/s]

Infill for DECT210 Waschmaschine at freq 10s performed earlier. Skipping.
Infill for DECT200 Waschmaschine at freq 10s performed earlier. Skipping.
Infill for Smart Switch 6 Spülmaschine at freq 10s performed earlier. Skipping.
Infill for DECT200 Spülmaschine at freq 10s performed earlier. Skipping.
